In [0]:
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
)

import os
# env = os.environ["S3PREFIX"] 
# Import the module
import sys
sys.path.append("../aws_utils")

import aws_utils
# Get Environment Prefix
 
env = aws_utils.get_env_prefix()
import delta_table_creator, transform

In [0]:
schema=(
        "DAgreementId string,"
        "SourceSystem string,"
        "PortfolioKey string,"
        "PortfolioName string,"
        "IngestionTimestamp timestamp,"
        "BatchId string,"
        "DAgreementKey string,"
        "AgreementKey string,"
        "AgreementStatusCode string,"
        "AgreementStatus string,"
        "IssueDate date,"
        "ExpiredDate date,"
        "SignedDate date,"
        "Currency string,"
        "LimitAmount decimal(19,2),"
        "LimitUtilizedAmount decimal(19,2),"
        "NumberOfRestructurings int,"
        "NumberOfRestructuresESMA int,"    
        "BusinessDate date"
)

partition_columns = ["BusinessDate"]
run_date = dbutils.widgets.get("run_date")
unity = dbutils.widgets.get("unity")
table_name = "DAgreement"
derivative_table_path = ""

# ================== Create Delta Table ==================
if unity == "True":
    derivative_table_path = f"`{env}-cdh`.derivativeactive.{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_unity_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
else:
    derivative_table_path = f"s3://{env}-ec1-cepal-derivative-zone/{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )


# ================== Delete From Delta Table ==================

transform.delete_from_unity_table(
    spark,
    derivative_table_path,
    run_date,  # type: ignore
)

In [0]:
export_date = '2025-02-17'

In [0]:
# Create temp view for each source table, based on the run date

agreement = spark.sql(f""" select *
FROM  `{env}-cdh`.primary.agreement
WHERE SourceSystem = 'profits'
AND BusinessDate = {run_date!r}
"""
) 

agreement.createOrReplaceTempView("agreement")

restructuring = spark.sql(f""" select 
DISTINCT NewAgreementKey,
RestructuringDate
FROM  `{env}-cdh`.primary.restructuring
WHERE (RestructuringDate > {export_date!r} and RestructuringDate <= {run_date!r})
"""
) 

restructuring.createOrReplaceTempView("restructuring")

exportmodificationspee2025 = spark.sql(f""" select *
FROM  `{env}-cdh`.reference.modifications_pee_20250217 mdp
"""
) 

exportmodificationspee2025.createOrReplaceTempView("exportmodificationspee2025")


exportmodificationspeeESMA = spark.sql(f""" select *
FROM  `{env}-cdh`.reference.modifications_pee_ESMA mdp
"""
) 

exportmodificationspeeESMA.createOrReplaceTempView("exportmodificationspeeESMA")

In [0]:
exposure_ever = spark.sql(f""" select
    ExposureKey,
    AgreementKey,
    SourceSystem,
    PortfolioKey,
    PortfolioName,
    IsActiveFlag,
    PrimaryPartyFlag,
    BusinessDate

FROM `{env}-cdh`.primary.Exposure
WHERE BusinessDate > {export_date!r} and BusinessDate <= {run_date!r}
"""
)  

exposure_ever.createOrReplaceTempView("exposure_ever")

exposure_profits = exposure_ever.filter((exposure_ever.SourceSystem == 'profits') & (exposure_ever.PrimaryPartyFlag == True))

exposure_profits.createOrReplaceTempView("exposure_profits")

SettlementExposure = spark.sql(f""" select
 
    Bucket,
    ExposureKey,
    SettlementKey,
    BusinessDate,
    SettlementStatus,
    isPrimarySettlementFlag,
    IsActiveFlag,
    ImplementationDate,
    SettlementType,
    SubmissionDate,
    SourceSystem

FROM `{env}-cdh`.primary.SettlementExposure
WHERE BusinessDate > {export_date!r} and BusinessDate <= {run_date!r}
AND ImplementationDate > {export_date!r} and ImplementationDate <= {run_date!r}
"""
)  

SettlementExposure.createOrReplaceTempView("SettlementExposure")

SettlementExposure_Bucket_qca = spark.sql(f"""
    select SettlementKey, AgreementKey, ImplementationDate
    from (
        select settlexp.SettlementKey, exposure.AgreementKey, ImplementationDate
        -- row_number() over (partition by exposure.ExposureKey order by SubmissionDate desc) as rown
        from SettlementExposure settlexp
        inner join exposure_profits exposure
        on substring(exposure.exposurekey, 4, 50) = settlexp.exposurekey
        where 1=1
        and settlexp.SourceSystem='qca'
        and SettlementType != 'Νόμος 3869 – Προσωρινή Διαταγή'
        and SettlementStatus IN ('Running', 'Partially Fulfilled')
        and exposure.ExposureKey IS NOT NULL
    ) a
    -- where rown=1
""")

SettlementExposure_Bucket_qc = spark.sql(f"""
    select settlexp.SettlementKey, AgreementKey, ImplementationDate
    from SettlementExposure settlexp
    inner join exposure_profits on
    settlexp.ExposureKey = exposure_profits.ExposureKey
    where 1=1
    and settlexp.SourceSystem='process'
    and settlexp.IsActiveFlag=1
    and settlexp.isPrimarySettlementFlag = True
    and COALESCE(settlexp.SettlementStatus, 'Cancelled') 
      NOT IN ('Client Cancelled', 'Rejected', 'Cancelled', 'Internal Cancelled', 'Cancelled - Internal Rejection')
""")

SettlementExposure_Bucket_qca.createOrReplaceTempView("SettlementExposure_Bucket_qca")

SettlementExposure_Bucket_qc.createOrReplaceTempView("SettlementExposure_Bucket_qc")

ActiveSettlementsqca = spark.sql(f""" 
    select DISTINCT AgreementKey, count(DISTINCT ImplementationDate) as settlementstotalqca
    from SettlementExposure_Bucket_qca
    group by AgreementKey                     
""")

ActiveSettlementsqc = spark.sql(f"""
    select DISTINCT AgreementKey, count(DISTINCT ImplementationDate) as settlementstotalqc
    from SettlementExposure_Bucket_qc
    group by AgreementKey                     
""")

ActiveSettlementsqca.createOrReplaceTempView("ActiveSettlementsqca")

ActiveSettlementsqc.createOrReplaceTempView("ActiveSettlementsqc")


In [0]:
# TBD: Remove the unecessary join with modification PEE 2025-01 and 2024-06. ONLY include the latest export. 

target_table_dagreement_pre = spark.sql(f""" 
SELECT
    a.AgreementId,
    a.SourceSystem,
    a.PortfolioKey,
    a.PortfolioName,
    a.BatchId,
    a.AgreementKey,
    a.AgreementStatusCode,
    a.AgreementStatus,
    a.IssueDate,
    a.ExpiredDate,
    a.SignedDate,
    a.Currency,
    a.LimitAmount,
    a.LimitUtilizedAmount,  
    SUM(
        CASE 
            WHEN (rs.RestructuringDate > {export_date!r} 
            ) THEN 1
            ELSE 0
        END
    ) OVER (PARTITION BY a.AgreementKey) + COALESCE(mdp2025.NumberofRestructurings,0) + COALESCE(settlementstotalqca, 0) + COALESCE(settlementstotalqc, 0)  AS TotalRestructurings,
    mdpesma.NumberOfRestructuresESMA,
    a.BusinessDate
FROM
    agreement a
LEFT JOIN exportmodificationspee2025 mdp2025
ON a.AgreementKey = mdp2025.AgreementKey 
LEFT JOIN exportmodificationspeeESMA mdpesma
ON a.AgreementKey = mdpesma.AgreementKey
LEFT JOIN restructuring rs 
ON a.AgreementKey = rs.NewAgreementKey
LEFT JOIN ActiveSettlementsqc actsettlqc
ON a.AgreementKey = actsettlqc.AgreementKey
LEFT JOIN ActiveSettlementsqca actsettllca
ON a.AgreementKey = actsettllca.AgreementKey
""")

target_table_dagreement_pre.createOrReplaceTempView('target_tabledagreement')

In [0]:
target_table_dagreementid_primary_key_fields = [
    agreement.AgreementId,
]

target_table_dagreementKey_primary_key_fields = [
    agreement.AgreementKey,
]

target_table_dagreement = (
    target_table_dagreement_pre.withColumn(
        "DAgreementId",
        sha2(
            concat_ws(";", *target_table_dagreementid_primary_key_fields),
            256,
        ),
    ).withColumn(
        "DAgreementKey",
        concat_ws(";", *target_table_dagreementKey_primary_key_fields)
    )
)


result_df = target_table_dagreement.select(
    target_table_dagreement.DAgreementId.cast("String"),
    target_table_dagreement.SourceSystem.cast("String"),
    target_table_dagreement.PortfolioKey.cast("String"),
    target_table_dagreement.PortfolioName.cast("String"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
    ),
    target_table_dagreement.BatchId.cast("String"),
    target_table_dagreement.DAgreementKey.cast("String"),
    target_table_dagreement.AgreementKey.cast("String"),
    target_table_dagreement.AgreementStatusCode.cast("String"),
    target_table_dagreement.AgreementStatus.cast("String"),
    target_table_dagreement.IssueDate.cast("Date"),
    target_table_dagreement.ExpiredDate.cast("Date"),
    target_table_dagreement.SignedDate.cast("Date"),
    target_table_dagreement.Currency.cast("String"),
    target_table_dagreement.LimitAmount.cast("Decimal(19,2)"),
    target_table_dagreement.LimitUtilizedAmount.cast("Decimal(19,2)"),
    target_table_dagreement.TotalRestructurings.cast("Int").alias("NumberOfRestructurings"),
    target_table_dagreement.NumberOfRestructuresESMA.cast("Int").alias("NumberOfRestructuresESMA"),
    target_table_dagreement.BusinessDate.cast("Date")
)


In [0]:
if unity == "True":
    result_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(derivative_table_path) # in order to write in delta_lake
else: 
    result_df.write \
        .format("delta") \
        .mode("append") \
        .save(derivative_table_path) # in order to write in delta_lake 